# Prepare RNA and Motif Data
Following the steps detailed in DynaVelo, prepare the ATAC/z-score data and RNA expression data. These can then be used for model training and evaluation of velocities.

In [2]:
import anndata as ad
import scanpy as sc 
import pandas as pd
import numpy as numpy

In [45]:
python_data_dir = r"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_testing\data"
r_data_dir = r"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_preprocessing\data"

# Load RNA total count data
total_counts_path = fr"{python_data_dir}\total_counts.h5ad"
adata_rna = sc.read_h5ad(total_counts_path)

# Load ATAC z-score data
zscore_path = fr"{r_data_dir}\Exp4_ctr_chromvar_data.csv"
data_atac = pd.read_csv(zscore_path, header=0, index_col=0).T
adata_atac = ad.AnnData(data_atac.values, obs=pd.DataFrame(index=data_atac.index), var=pd.DataFrame(index=data_atac.columns))
adata_atac.write_h5ad(fr"{python_data_dir}\z_scores_raw.h5ad")

In [47]:
# Ensure that both datasets have the same cells
rna_cells = adata_rna.obs_names
atac_cells = adata_atac.obs_names

intersection_cells = rna_cells.intersection(atac_cells)

adata_rna = adata_rna[intersection_cells, :]  # Already in cells-by-gene
adata_atac = adata_atac[intersection_cells, :]  # Convert to cells-by-peak

adata_atac.write_h5ad(fr"{python_data_dir}\z_scores_intersect.h5ad")
adata_rna.write_h5ad(fr"{python_data_dir}\total_counts_intersect.h5ad")

In [48]:
adata_atac

View of AnnData object with n_obs × n_vars = 5380 × 746

In [49]:
adata_rna

View of AnnData object with n_obs × n_vars = 5380 × 32285
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'S.Score', 'G2M.Score', 'Phase', 'SCT_snn_res.0.8', 'seurat_clusters', 'doublets_normal_cluster_based', 'doublets_random', 'doublets_RNA_consensus', 'nCount_SCT_no_regression', 'nFeature_SCT_no_regression', 'tricyclePosition', 'Phase_tricycle', 'tricycle_fine', 'cell_cycle_atac_classification', 'PromoterPercent', 'immunophenotype_predicted', 'predicted.id', 'prediction.score.LT', 'prediction.score.ST', 'prediction.score.MPP', 'prediction.score.LS_K', 'prediction.score.max', 'Identity_for_diff_testing', 'RF_prediction', 'angle_degree', 'immunophenotype_predicted_cleaned', 'HSC.promoters', 'LMPP.promoters', 'myeloid.promoters', 'MEP.promoters',

In [ ]:
# Add spliced and unspliced layers to the RNA AnnData object

In [ ]:
# Filter RNA data
sc.pp.filter_cells(adata_rna, min_counts=1000)
adata_rna.obs['read_depth_log10'] = np.log10(adata_rna.obs['n_counts'])

adata_rna = adata_rna[:, ~adata_rna.var_names.isin(['Malat1'])]
scv.pp.filter_genes(adata_rna, min_shared_counts=10)

median_x = np.median(adata_rna.X.toarray().sum(1))
print(f'Median of read depth in X: {median_x}')
sc.pp.normalize_total(adata_rna, target_sum=median_x)

median_s = np.median(adata_rna.layers['spliced'].toarray().sum(1))
print(f'Median of read depth in spliced: {median_s}')
sc.pp.normalize_total(adata_rna, target_sum=median_s, layer='spliced')

median_u = np.median(adata_rna.layers['unspliced'].toarray().sum(1))
print(f'Median of read depth in unspliced: {median_u}')
sc.pp.normalize_total(adata_rna, target_sum=median_u, layer='unspliced')

sc.pp.log1p(adata_rna)
sc.pp.highly_variable_genes(adata_rna, n_top_genes=2000, flavor='seurat', batch_key='Sample')
adata_rna.layers["X_log"] = adata_rna.X.copy()